#TODO Ainda é necessario tratar os horarios de chamados e tempo de atendimento.

In [0]:
from config import BRONZE_SCHEMA,SILVER_SCHEMA, CATALOG
from utils import dq_check, dq_check_unique
from pyspark.sql import functions as F

In [0]:
bronze_chamados = spark.table(f"{BRONZE_SCHEMA}.chamados")
bronze_chamados.printSchema()
display(bronze_chamados)

In [0]:
dq_check_unique('chamados_bronze','unique_id',bronze_chamados,['id_chamado'])
dq_check_unique('chamados_bronze','unique_id',bronze_chamados,['id_chamado','id_cliente'])
dq_check_unique('chamados_bronze','unique_id',bronze_chamados,['id_chamado','id_cliente','canal'])
dq_check_unique('chamados_bronze','unique_id',bronze_chamados,['id_chamado','id_cliente','canal','id_atendente'])
dq_check_unique('chamados_bronze','unique_id',bronze_chamados,['id_chamado','id_cliente','canal','id_atendente','resolvido'])
dq_check_unique('chamados_bronze','unique_id',bronze_chamados,['id_chamado','id_cliente','canal','id_atendente','resolvido','motivo'])

In [0]:
chamados_sem_duplicados = bronze_chamados.drop('ingestion_datetime').dropDuplicates()
#linhas duplicadas 1 para 1 
_,passou = dq_check_unique('chamados_bronze','unique_id',chamados_sem_duplicados,['id_chamado','id_cliente'])
if not passou:
    raise ValueError("Existem registros com mesma chave composta de id_chamado e id_cliente")

In [0]:
silver_chamados= (
    chamados_sem_duplicados 
    .withColumn(
        'nome_motivo',
        F.lower(F.expr("trim(motivo) COLLATE UNICODE_CI_AI"
        )),
    )
    .drop('motivo')
)
display(silver_chamados.groupBy('nome_motivo').count().orderBy('count'))

In [0]:
#removendo o motivo e utilizando somento o id. afim de normalizar o modelo relacional
silver_motivos = spark.table(f"{SILVER_SCHEMA}.base_motivos")
motivos= (
    silver_motivos.select('nome_motivo','id_motivo')
    .select('id_motivo','nome_motivo')
    .withColumn(
        'nome_motivo',
        F.expr("trim(nome_motivo) COLLATE UNICODE_CI_AI"),
    )
)
#
silver_chamados=(
    silver_chamados
    .join(motivos, on ='nome_motivo', how='left')
    .drop('nome_motivo')
)
display(silver_chamados)

In [0]:
subs={
    '_': ' ',
    '.': '',
    'atend': 'atendimento',
    'chat bot': 'chatbot'
}
parciais =['atendimento']
def string_mapping(df,column,subs:dict,parciais:list):
    import re

    df_mapped = (
        df.withColumn(column, F.lower(F.trim(F.col(f'{column}'))),
                      )
    )
    expressao = F.col(column)
    for original, substituto in subs.items():
        expressao= F.regexp_replace(
            expressao, re.escape(original), substituto
        )
    df_mapped = df_mapped.withColumn(column, expressao)
    for parcial in parciais:
        df_mapped = df_mapped.withColumn(column,
                                     F.regexp_replace(
                                         F.col(column),
                                         rf"{parcial}\S*",
                                         parcial
                                     ))
    return df_mapped

df_ch = string_mapping(silver_chamados,'canal',subs,parciais)
display(df_ch.groupBy('canal').count())


In [0]:
silver_chamados = df_ch

In [0]:
#aqui eu acho que um mapeamento direto e simples é o correto devido as poucas possibilidades
subs ={
    'nao resolvido':'0',
    'nao': '0',
    'não': '0',
    'n':'0',
    '00':'0',
    'sim':'1',
    's': '1',

}
parciais=['1','0']
df_ch = string_mapping(silver_chamados,'resolvido',subs,parciais)
display(df_ch.groupBy('resolvido').count())

In [0]:
df_ch =(df_ch.withColumn('resolvido', 
                 F.col('resolvido').try_cast('int'))
                 .filter(F.col('resolvido').between(0,1) | (F.col('resolvido').isNull()))
)
silver_chamados = df_ch
#Ainda existem valores nulos


In [0]:
(silver_chamados.groupBy('hora_finalizacao_atendimento').count()).show()
(silver_chamados.groupBy('hora_abertura_chamado').count()).show()
silver_chamados = silver_chamados.drop('hora_abertura_chamado','hora_finalizacao_atendimento')


In [0]:
silver_chamados = silver_chamados.withColumn(
    "tempo_espera",
    F.when(F.col("tempo_espera") == "NULL", F.lit(None))
     .otherwise(F.col("tempo_espera")),
)
silver_chamados = silver_chamados.withColumn(
    "tempo_atendimento",
    F.when(F.col("tempo_atendimento") == "NULL", F.lit(None))
     .otherwise(F.col("tempo_atendimento")),
) 

In [0]:
silver_chamados = silver_chamados.withColumn(
    "tempo_espera",
    F.expr("try_cast(tempo_espera AS DOUBLE)"),
)
silver_chamados = silver_chamados.withColumn(
    "tempo_atendimento",
    F.expr("try_cast(tempo_atendimento AS DOUBLE)"),
)
silver_chamados = silver_chamados.withColumn(
    'tempo_espera',
    F.when(F.col('hora_inicio_atendimento') =='igual a hora de abertura',F.lit(0)).otherwise(F.col('tempo_espera'))
)
silver_chamados= silver_chamados.drop('hora_inicio_atendimento')
display(silver_chamados)

In [0]:
#silver_chamados_hora = spark.table(f"{SILVER_SCHEMA}.chamados_hora")